In [15]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import ipywidgets as widgets
from IPython.display import display, clear_output

In [16]:
# 1. LOAD YOUR DATA
df = pd.read_csv('hvac_optimization_data.csv')

In [17]:
# 2. TRAIN THE DECISION TREE
X = df[['Occupancy_Count', 'Ambient_Temp_C']]
y = df['Required_Cooling_kW']

In [18]:
# Splitting data to calculate real-world accuracy
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = DecisionTreeRegressor(max_depth=5, random_state=42)
model.fit(X_train, y_train)

# Calculate Accuracy Metrics
preds = model.predict(X_test)
r2 = r2_score(y_test, preds)
mae = mean_absolute_error(y_test, preds)

In [24]:
# 3. DASHBOARD RENDERING
output = widgets.Output()

def update_dashboard(log_index):
    with output:
        clear_output(wait=True)

        # Data slice for the Heatmap
        snapshot = df.iloc[log_index:log_index+5].copy()
        snapshot['Predicted_kW'] = model.predict(snapshot[['Occupancy_Count', 'Ambient_Temp_C']])

        print(f"--- ANALYTICS REPORT ---")
        print(f"Model R² Accuracy: {r2:.4f} | Mean Absolute Error: {mae:.2f} kW\n")

        # GRAPH 1: Zone Heatmap (Predicted Load)
        fig_heat = px.imshow(
            [snapshot['Predicted_kW'].values],
            labels=dict(x="Zones", color="Load (kW)"),
            x=snapshot['Zone_ID'].values,
            color_continuous_scale='Viridis',
            title="Spatial Heatmap: Predicted Cooling Demand"
        )
        fig_heat.update_layout(height=250, margin=dict(t=40, b=10))

        # GRAPH 2: Regression Analysis (Actual vs Predicted)
        fig_acc = go.Figure()
        fig_acc.add_trace(go.Scatter(x=y_test, y=preds, mode='markers', name='Data Points', marker=dict(color='#1f77b4')))
        fig_acc.add_trace(go.Scatter(x=[y_test.min(), y_test.max()], y=[y_test.min(), y_test.max()],
                                   mode='lines', name='Perfect Accuracy', line=dict(color='red', dash='dash')))
        fig_acc.update_layout(title="Model Accuracy (Actual vs. Predicted)", xaxis_title="Actual kW", yaxis_title="Predicted kW", height=350)

        # GRAPH 3: Feature Importance
        importance = pd.DataFrame({'Feature': X.columns, 'Importance': model.feature_importances_})
        fig_feat = px.bar(importance, x='Feature', y='Importance', title="Key Decision Drivers", color='Importance')
        fig_feat.update_layout(height=300)

        # DISPLAY ALL
        fig_heat.show()
        fig_acc.show()
        fig_feat.show()

In [25]:
# 4. CONTROLS
log_slider = widgets.IntSlider(
    value=0, min=0, max=len(df)-5, step=5,
    description='System Log:',
    layout=widgets.Layout(width='70%')
)

widgets.interactive(update_dashboard, log_index=log_slider)

display(widgets.HTML("<h2 style='text-align:center; color:#1A5276;'>HVAC Optimization & Prediction Dashboard</h2>"))
display(log_slider, output)

HTML(value="<h2 style='text-align:center; color:#1A5276;'>HVAC Optimization & Prediction Dashboard</h2>")

IntSlider(value=0, description='System Log:', layout=Layout(width='70%'), max=95, step=5)

Output()